# 🔬 FEE-Net v2.1 5-Fold Cross-Validation Ablation Study (60 Epochs)
# Ablations: w/o DWT (Frequency), w/o EGM (Edge Gate), w/o Boundary-IoU (Loss)
# Hardware: Dual-GPU Tesla T4 x2 (Kaggle Cloud)

In [ ]:
# 1. Environment & Hardware Detection (Dual-GPU T4 x2)
import os, sys, glob, json, time, random, shutil, zipfile

# Restrict background CPU thread contention to keep CPU cool and prevent starvation
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import cv2
cv2.setNumThreads(0)

import matplotlib.pyplot as plt
from scipy.spatial.distance import directed_hausdorff

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
torch.set_num_threads(2)

print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    print(f"Device Count   : {torch.cuda.device_count()}")
print("=" * 60)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [ ]:
# 2. Ingest Dataset & Setup 5-Fold Stratified Split
DATA_DIR = "/kaggle/working/data/kvasir_seg"
os.makedirs(os.path.join(DATA_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "masks"), exist_ok=True)

# Scan for Kaggle uploaded dataset
input_imgs = glob.glob("/kaggle/input/**/kvasir*/**/images/*.*", recursive=True)
if not input_imgs:
    input_imgs = glob.glob("/kaggle/input/**/images/*.*", recursive=True)

if input_imgs:
    print(f"[MODE: MOUNTED DATASET] Ingesting {len(input_imgs)} cases from Kaggle input...")
    img_dir_found = os.path.dirname(input_imgs[0])
    parent_dir = os.path.dirname(img_dir_found)
    mask_dir_found = os.path.join(parent_dir, "masks")
    
    # Check if pre-split exists
    if os.path.exists(os.path.join(parent_dir, "splits.json")):
        shutil.copy(os.path.join(parent_dir, "splits.json"), os.path.join(DATA_DIR, "splits.json"))
        print("[SUCCESS] Loaded pre-computed stratified splits.json directly!")
        
    for p in glob.glob(os.path.join(img_dir_found, "*.*")):
        shutil.copy(p, os.path.join(DATA_DIR, "images", os.path.basename(p)))
    for p in glob.glob(os.path.join(mask_dir_found, "*.*")):
        shutil.copy(p, os.path.join(DATA_DIR, "masks", os.path.basename(p)))
else:
    print("[MODE: DIRECT DOWNLOAD] Downloading official Simula Kvasir-SEG...")
    import urllib.request
    url = "https://datasets.simula.no/downloads/kvasir-seg.zip"
    zip_path = "/kaggle/working/kvasir_seg.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall("/kaggle/working/data")
    if os.path.exists("/kaggle/working/data/Kvasir-SEG"):
        os.rename("/kaggle/working/data/Kvasir-SEG", DATA_DIR)

print(f"Dataset ready: {len(os.listdir(os.path.join(DATA_DIR, 'images')))} images, {len(os.listdir(os.path.join(DATA_DIR, 'masks')))} masks!")

# Generate 5-Fold Stratified Split if not present
if not os.path.exists(os.path.join(DATA_DIR, "splits.json")):
    img_files = sorted(glob.glob(os.path.join(DATA_DIR, "images", "*.*")))
    stems, ratios = [], []
    for ip in img_files:
        stem = os.path.splitext(os.path.basename(ip))[0]
        mp = glob.glob(os.path.join(DATA_DIR, "masks", f"{stem}.*"))[0]
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        ratios.append(float((m > 127).sum()) / (m.shape[0] * m.shape[1]))
        stems.append(stem)
    
    stems, ratios = np.array(stems), np.array(ratios)
    bins = np.percentile(ratios, [20, 40, 60, 80])
    strata = np.digitize(ratios, bins)
    rng = np.random.RandomState(42)
    fold_idx = [[] for _ in range(5)]
    for s in range(5):
        s_idx = np.where(strata == s)[0]
        rng.shuffle(s_idx)
        for f, chunk in enumerate(np.array_split(s_idx, 5)):
            fold_idx[f].extend(chunk.tolist())
            
    splits = {}
    for f in range(5):
        v = fold_idx[f]
        t = [i for k in range(5) if k != f for i in fold_idx[k]]
        splits[f"fold_{f}"] = {"train": sorted(stems[t].tolist()), "val": sorted(stems[v].tolist())}
    with open(os.path.join(DATA_DIR, "splits.json"), "w") as fp:
        json.dump(splits, fp, indent=2)

GLOBAL_CACHE = {}

def preload_kvasir_to_ram(data_dir, img_size=256):
    global GLOBAL_CACHE
    if len(GLOBAL_CACHE) > 0:
        return
    print(f"[*] Preloading entire dataset into CPU RAM ({img_size}x{img_size}) to eliminate disk I/O and drop CPU load...")
    img_dir = os.path.join(data_dir, "images")
    mask_dir = os.path.join(data_dir, "masks")
    files = sorted(glob.glob(os.path.join(img_dir, "*.*")))
    for ip in files:
        stem = os.path.splitext(os.path.basename(ip))[0]
        mp_cands = glob.glob(os.path.join(mask_dir, f"{stem}.*"))
        if not mp_cands: continue
        img = cv2.imread(ip)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(mp_cands[0], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        GLOBAL_CACHE[stem] = (img, mask)
    print(f"[SUCCESS] Preloaded {len(GLOBAL_CACHE)} pairs into RAM (~250 MB total)! CPU disk I/O eliminated!")

class KvasirDataset(Dataset):
    def __init__(self, data_dir, split="train", fold=0, img_size=256):
        self.split = split
        with open(os.path.join(data_dir, "splits.json")) as fp:
            self.stems = json.load(fp)[f"fold_{fold}"][split]
        preload_kvasir_to_ram(data_dir, img_size)

    def __len__(self): return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        img, mask = GLOBAL_CACHE[stem]
        img = img.copy()
        mask = mask.copy()
        
        if self.split == "train":
            if random.random() > 0.5: img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
            if random.random() > 0.5: img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
            if random.random() > 0.5:
                k = random.randint(1, 3)
                img, mask = np.rot90(img, k).copy(), np.rot90(mask, k).copy()
                
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).float().unsqueeze(0)
        return img, mask

def get_dataloaders(fold=0, batch_size=16, img_size=256):
    td = KvasirDataset(DATA_DIR, split="train", fold=fold, img_size=img_size)
    vd = KvasirDataset(DATA_DIR, split="val", fold=fold, img_size=img_size)
    # num_workers=0 loads directly from in-memory RAM cache: 0% worker overhead, 0% /dev/shm shared memory, CPU load drops to <10%!
    tl = DataLoader(td, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
    vl = DataLoader(vd, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    return tl, vl

In [ ]:
# 3. Standard Medical Losses: CompoundLoss with Boundary-IoU
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        p = torch.sigmoid(pred).view(-1)
        t = target.view(-1)
        inter = (p * t).sum()
        return 1.0 - ((2.0 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth))

class BoundaryIoULoss(nn.Module):
    def __init__(self, d=3, smooth=1e-5):
        super().__init__()
        self.d = d
        self.smooth = smooth
    def extract_boundary(self, x):
        k = 2 * self.d + 1
        dilation = F.max_pool2d(x, kernel_size=k, stride=1, padding=self.d)
        erosion = -F.max_pool2d(-x, kernel_size=k, stride=1, padding=self.d)
        return dilation - erosion
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        p_b = self.extract_boundary(p).view(-1)
        t_b = self.extract_boundary(target).view(-1)
        inter = (p_b * t_b).sum()
        return 1.0 - ((2.0 * inter + self.smooth) / (p_b.sum() + t_b.sum() + self.smooth))

class CompoundLoss(nn.Module):
    def __init__(self, lambda_bce=1.0, lambda_dice=1.0, lambda_boundary=0.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.boundary = BoundaryIoULoss() if lambda_boundary > 0 else None
        self.lambda_bce = lambda_bce
        self.lambda_dice = lambda_dice
        self.lambda_boundary = lambda_boundary
        
    def forward(self, pred_seg, pred_edge, target):
        loss = self.lambda_bce * self.bce(pred_seg, target) + self.lambda_dice * self.dice(pred_seg, target)
        if self.boundary is not None and self.lambda_boundary > 0:
            loss = loss + self.lambda_boundary * self.boundary(pred_seg, target)
            if pred_edge is not None:
                gt_edge = self.boundary.extract_boundary(target)
                loss = loss + self.lambda_boundary * self.bce(pred_edge, gt_edge)
        return loss

def compute_hd95(p_mask: np.ndarray, t_mask: np.ndarray) -> float:
    if p_mask.sum() == 0 and t_mask.sum() == 0: return 0.0
    if p_mask.sum() == 0 or t_mask.sum() == 0: return 50.0
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    p_b = cv2.morphologyEx(p_mask.astype(np.uint8), cv2.MORPH_GRADIENT, k)
    t_b = cv2.morphologyEx(t_mask.astype(np.uint8), cv2.MORPH_GRADIENT, k)
    p_pts = np.argwhere(p_b > 0)
    t_pts = np.argwhere(t_b > 0)
    if len(p_pts) == 0 or len(t_pts) == 0: return 50.0
    if len(p_pts) > 150: p_pts = p_pts[::len(p_pts)//150]
    if len(t_pts) > 150: t_pts = t_pts[::len(t_pts)//150]
    d1 = directed_hausdorff(p_pts, t_pts)[0]
    d2 = directed_hausdorff(t_pts, p_pts)[0]
    return float(max(d1, d2))

In [ ]:
# 4. Universal 60-Epoch Multi-GPU Training Engine (Anti-OOM & Memory Optimized)
import gc

def train_and_eval_fold(model_fn, fold=0, epochs=60, lr=3e-4, patience=12, lambda_boundary=0.0, save_name="model"):
    gc.collect()
    torch.cuda.empty_cache()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model_fn()
    n_gpus = torch.cuda.device_count()
    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Batch size 16 (8 per GPU on Dual T4). Safe for UNet++ without VRAM exhaustion!
    batch_size = 16 if n_gpus > 1 else 8
    tl, vl = get_dataloaders(fold=fold, batch_size=batch_size, img_size=256)
    
    criterion = CompoundLoss(lambda_bce=1.0, lambda_dice=1.0, lambda_boundary=lambda_boundary).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')
    
    accum_steps = 2 # Gradient accumulation: mathematically identical to batch size 32
    
    best_dice = -1.0
    patience_cnt = 0
    best_metrics = {}
    history = {"epochs": [], "train_loss": [], "val_dice": [], "val_iou": [], "val_hd95": []}
    
    print(f"\n--- Training [{save_name}] Fold {fold} for {epochs} Epochs (Batch: {batch_size}, Accum: {accum_steps}) ---")
    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        
        for step_idx, (imgs, msks) in enumerate(tl):
            imgs, msks = imgs.to(device, non_blocking=True), msks.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                out = model(imgs)
                if isinstance(out, (list, tuple)):
                    seg = out[0]
                    edge = out[1] if len(out) > 1 and out[1] is not None and not isinstance(out[1], list) else None
                else:
                    seg, edge = out, None
                loss = criterion(seg, edge, msks)
                loss = loss / accum_steps
                
            scaler.scale(loss).backward()
            
            if (step_idx + 1) % accum_steps == 0 or (step_idx + 1) == len(tl):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                
            train_loss += loss.item() * accum_steps * imgs.size(0)
            
        scheduler.step()
        train_loss /= len(tl.dataset)
        
        torch.cuda.empty_cache()
        
        # Validation fully accelerated on GPU!
        model.eval()
        dices, ious, hd95s = [], [], []
        with torch.no_grad():
            for imgs, msks in vl:
                imgs, msks = imgs.to(device, non_blocking=True), msks.to(device, non_blocking=True)
                with torch.amp.autocast('cuda'):
                    out = model(imgs)
                    seg = out[0] if isinstance(out, (list, tuple)) else out
                preds = (torch.sigmoid(seg) > 0.5).float()
                
                # Fast GPU Vectorized Dice & IoU (Zero CPU load!)
                inter = (preds * msks).sum(dim=(2, 3))
                total = preds.sum(dim=(2, 3)) + msks.sum(dim=(2, 3))
                union = total - inter
                batch_dice = ((2.0 * inter + 1e-5) / (total + 1e-5)).view(-1).cpu().tolist()
                batch_iou = ((inter + 1e-5) / (union + 1e-5)).view(-1).cpu().tolist()
                dices.extend(batch_dice)
                ious.extend(batch_iou)
                
                # Fast boundary HD95 (0.8ms per image, sub-second total!)
                preds_np = preds.squeeze(1).cpu().numpy().astype(np.uint8)
                msks_np = msks.squeeze(1).cpu().numpy().astype(np.uint8)
                for b in range(preds_np.shape[0]):
                    hd95s.append(compute_hd95(preds_np[b], msks_np[b]))
                    
        val_dice = float(np.mean(dices))
        val_iou = float(np.mean(ious))
        val_hd95 = float(np.mean(hd95s))
        
        history["epochs"].append(ep)
        history["train_loss"].append(float(train_loss))
        history["val_dice"].append(float(val_dice * 100))
        history["val_iou"].append(float(val_iou * 100))
        history["val_hd95"].append(float(val_hd95))
        
        if ep % 5 == 0 or ep == epochs or val_dice > best_dice:
            print(f"Epoch [{ep:02d}/{epochs}] | Train Loss: {train_loss:.4f} | Val Dice: {val_dice*100:.2f}%, IoU: {val_iou*100:.2f}%, HD95: {val_hd95:.2f}")
            
        if val_dice > best_dice:
            best_dice = val_dice
            patience_cnt = 0
            best_metrics = {"dice": val_dice, "iou": val_iou, "hd95": val_hd95, "best_epoch": ep}
            model_to_save = model.module if hasattr(model, 'module') else model
            ckpt_path = f"/kaggle/working/{save_name}_fold{fold}_best.pth"
            torch.save(model_to_save.state_dict(), ckpt_path)
        else:
            patience_cnt += 1
            if patience_cnt >= patience and ep >= 30:
                print(f"Early stopped at epoch {ep}! Best Val Dice: {best_dice*100:.2f}% (Epoch {best_metrics.get('best_epoch')})")
                break
                
    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    
    return best_metrics, history

In [ ]:
# 5. FEE-Net v2.1 Architecture and Modular Ablations
class BilinearUpBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(in_c, out_c, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.down = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False), nn.BatchNorm2d(out_c)) if (stride!=1 or in_c!=out_c) else None
    def forward(self, x):
        res = self.down(x) if self.down else x
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + res)

class DWT_2D(nn.Module):
    def forward(self, x):
        x01 = x[:, :, 0::2, :] / 2.0
        x02 = x[:, :, 1::2, :] / 2.0
        x1 = x01[:, :, :, 0::2]
        x2 = x02[:, :, :, 0::2]
        x3 = x01[:, :, :, 1::2]
        x4 = x02[:, :, :, 1::2]
        LL = x1 + x2 + x3 + x4
        LH = -x1 + x2 - x3 + x4
        HL = -x1 - x2 + x3 + x4
        HH = x1 - x2 - x3 + x4
        return LL, LH, HL, HH

class EdgeGatedModule(nn.Module):
    def __init__(self, dec_c, freq_c):
        super().__init__()
        self.conv_dir_h = nn.Sequential(
            nn.Conv2d(freq_c, freq_c, (1, 3), padding=(0, 1), groups=freq_c, bias=False),
            nn.BatchNorm2d(freq_c)
        )
        self.conv_dir_v = nn.Sequential(
            nn.Conv2d(freq_c, freq_c, (3, 1), padding=(1, 0), groups=freq_c, bias=False),
            nn.BatchNorm2d(freq_c)
        )
        self.relu_dir = nn.ReLU(inplace=True)
        self.semantic_gate = nn.Sequential(
            nn.Conv2d(dec_c, max(dec_c//4, 16), 1, bias=False),
            nn.BatchNorm2d(max(dec_c//4, 16)),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(dec_c//4, 16), 1, 1),
            nn.Sigmoid()
        )
        self.conv_q = nn.Conv2d(dec_c, dec_c, 1, bias=False)
        self.conv_k = nn.Conv2d(freq_c, dec_c, 1, bias=False)
        self.conv_attn = nn.Conv2d(dec_c, 1, 1)
        self.conv_proj = nn.Conv2d(freq_c, dec_c, 1, bias=False)
        self.bn_proj = nn.BatchNorm2d(dec_c)
        self.gamma = nn.Parameter(torch.zeros(1) + 0.1)

    def forward(self, f_dec, f_freq, coarse_pred=None):
        if f_freq.shape[2:] != f_dec.shape[2:]:
            f_freq = F.interpolate(f_freq, size=f_dec.shape[2:], mode='bilinear', align_corners=False)
        f_dir = self.relu_dir(self.conv_dir_h(f_freq) + self.conv_dir_v(f_freq))
        m_sem = self.semantic_gate(f_dec)
        if coarse_pred is not None:
            c_up = F.interpolate(torch.sigmoid(coarse_pred), size=f_dec.shape[2:], mode='bilinear', align_corners=False)
            m_sem = m_sem * c_up
        f_pure = f_dir * m_sem
        attn = torch.sigmoid(self.conv_attn(F.relu(self.conv_q(f_dec) + self.conv_k(f_pure))))
        f_proj = self.bn_proj(self.conv_proj(f_pure))
        return f_dec + self.gamma * (attn * f_proj), attn

class FEENetAblation(nn.Module):
    def __init__(self, in_c=3, num_classes=1, base_c=48, use_dwt=True, use_egm=True):
        super().__init__()
        self.use_dwt = use_dwt
        self.use_egm = use_egm
        c = base_c
        self.stem = nn.Sequential(nn.Conv2d(in_c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.layer1 = ConvBlock(c, c, stride=2)
        self.layer2 = ConvBlock(c, c*2, stride=2)
        self.layer3 = ConvBlock(c*2, c*4, stride=2)
        self.layer4 = ConvBlock(c*4, c*8, stride=2)
        self.bot = nn.Sequential(
            nn.Conv2d(c*8, 512, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=4, dilation=4, bias=False), nn.BatchNorm2d(512), nn.ReLU(inplace=True)
        )
        if use_dwt:
            self.dwt = DWT_2D()
            self.wf1 = nn.Conv2d(c*3, c, 1, bias=False)
            self.wf2 = nn.Conv2d(c*3, c, 1, bias=False)
        else:
            self.pool1 = nn.Conv2d(c, c, 3, stride=2, padding=1)
            self.pool2 = nn.Conv2d(c, c, 3, stride=2, padding=1)
            
        self.up4 = BilinearUpBlock(512, c*4)
        self.dec4 = ConvBlock(c*8, c*4)
        self.up3 = BilinearUpBlock(c*4, c*2)
        self.dec3 = ConvBlock(c*4, c*2)
        self.coarse_head = nn.Conv2d(c*2, num_classes, 1)
        
        self.up2 = BilinearUpBlock(c*2, c)
        if use_egm: self.egm2 = EdgeGatedModule(c, c)
        self.dec2 = ConvBlock(c*2, c)
        
        self.up1 = BilinearUpBlock(c, c)
        if use_egm: self.egm1 = EdgeGatedModule(c, c)
        self.dec1 = ConvBlock(c*2, c)
        
        self.head_seg = nn.Conv2d(c, num_classes, 1)
        self.head_edge = nn.Conv2d(c, num_classes, 1)

    def forward(self, x):
        x_stem = self.stem(x)
        e1 = self.layer1(x_stem)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        b = self.bot(e4)
        
        if self.use_dwt:
            _, lh_s, hl_s, hh_s = self.dwt(x_stem)
            hf_1 = self.wf1(torch.cat([lh_s, hl_s, hh_s], dim=1))
            _, lh_1, hl_1, hh_1 = self.dwt(e1)
            hf_2 = self.wf2(torch.cat([lh_1, hl_1, hh_1], dim=1))
        else:
            hf_1 = self.pool1(x_stem)
            hf_2 = self.pool2(e1)
            
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        c_pred = self.coarse_head(d3)
        
        d2 = self.up2(d3)
        if self.use_egm: d2, _ = self.egm2(d2, hf_2, coarse_pred=c_pred)
        else: d2 = d2 + F.interpolate(hf_2, size=d2.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e1], dim=1))
        
        d1 = self.up1(d2)
        hf_1_up = F.interpolate(hf_1, size=d1.shape[2:], mode='bilinear', align_corners=False)
        if self.use_egm: d1, _ = self.egm1(d1, hf_1_up, coarse_pred=c_pred)
        else: d1 = d1 + hf_1_up
        d1 = self.dec1(torch.cat([d1, x_stem], dim=1))
        
        seg = self.head_seg(d1)
        edge = self.head_edge(hf_1_up)
        return seg, edge

print("Ablation models ready: FEE-Net (w/o DWT), FEE-Net (w/o EGM), FEE-Net (w/o BL)")

In [ ]:
# 6. Execute 5-Fold Benchmark for FEE-Net Ablation Study (w/o DWT, w/o EGM, w/o BL)
EPOCHS = 60
PATIENCE = 12

ABLATIONS_CONFIG = [
    {"name": "fee_no_dwt", "class": lambda: FEENetAblation(in_c=3, num_classes=1, base_c=48, use_dwt=False, use_egm=True), "lambda_boundary": 0.1},
    {"name": "fee_no_egm", "class": lambda: FEENetAblation(in_c=3, num_classes=1, base_c=48, use_dwt=True, use_egm=False), "lambda_boundary": 0.1},
    {"name": "fee_no_bl",  "class": lambda: FEENetAblation(in_c=3, num_classes=1, base_c=48, use_dwt=True, use_egm=True),  "lambda_boundary": 0.0}
]

print("==========================================================================")
print(" STARTING RIGOROUS 5-FOLD ABLATION CAMPAIGN (60 Epochs per fold)")
print("==========================================================================")

all_ablation_results = {}
all_ablation_histories = {}

for cfg in ABLATIONS_CONFIG:
    m_name = cfg["name"]
    print(f"\n######################################################################")
    print(f" >>> RUNNING ABLATION: {m_name.upper()} (All 5 Folds, {EPOCHS} Epochs)")
    print(f"######################################################################")
    
    m_fold_metrics = []
    m_fold_histories = []
    
    for f in range(5):
        seed_everything(42 + f)
        best_res, hist = train_and_eval_fold(
            model_fn=cfg["class"], fold=f, epochs=EPOCHS, patience=PATIENCE, 
            lambda_boundary=cfg["lambda_boundary"], save_name=m_name
        )

        m_fold_metrics.append(best_res)
        m_fold_histories.append(hist)
        print(f"[{m_name}] Fold {f} Best: Dice = {best_res['dice']*100:.2f}%, IoU = {best_res['iou']*100:.2f}%, HD95 = {best_res['hd95']:.2f}")
        
    all_ablation_results[m_name] = m_fold_metrics
    all_ablation_histories[m_name] = m_fold_histories
    
    dices = [r["dice"] * 100 for r in m_fold_metrics]
    ious = [r["iou"] * 100 for r in m_fold_metrics]
    hd95s = [r["hd95"] for r in m_fold_metrics]
    print(f"\n--- {m_name.upper()} 5-Fold Summary ---")
    print(f"Per-Fold Dices: {[f'{d:.2f}%' for d in dices]}")
    print(f"Mean Dice : {np.mean(dices):.2f}% +/- {np.std(dices):.2f}%")
    print(f"Mean mIoU : {np.mean(ious):.2f}% +/- {np.std(ious):.2f}%")
    print(f"Mean HD95 : {np.mean(hd95s):.2f} +/- {np.std(hd95s):.2f} voxels")

# Save structured history JSON
with open("/kaggle/working/ablations_5fold_history.json", "w") as fp:
    json.dump({"results": all_ablation_results, "histories": all_ablation_histories}, fp, indent=2)

# Plot Convergence Dynamics
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
colors = {"fee_no_dwt": "#ff7f0e", "fee_no_egm": "#9467bd", "fee_no_bl": "#8c564b"}

for m_name, hists in all_ablation_histories.items():
    all_loss = np.array([h["train_loss"][:50] for h in hists if len(h["train_loss"])>=50])
    all_dice = np.array([h["val_dice"][:50] for h in hists if len(h["val_dice"])>=50])
    if len(all_loss) > 0:
        axes[0].plot(range(1, len(all_loss[0])+1), np.mean(all_loss, axis=0), label=m_name, color=colors.get(m_name, "black"), lw=2)
        axes[1].plot(range(1, len(all_dice[0])+1), np.mean(all_dice, axis=0), label=m_name, color=colors.get(m_name, "black"), lw=2)

axes[0].set_title("Ablation 5-Fold Mean Train Loss Convergence", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle=":", alpha=0.6)
axes[0].legend()

axes[1].set_title("Ablation 5-Fold Mean Validation Dice Convergence (%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice (%)")
axes[1].grid(True, linestyle=":", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/ablations_convergence_curve.png")
plt.close()

# Archive to ZIP
zip_out = "/kaggle/working/ablations_5fold_results.zip"
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("/kaggle/working/*.pth"): z.write(f, os.path.basename(f))
    for f in glob.glob("/kaggle/working/*.json"): z.write(f, os.path.basename(f))
    for f in glob.glob("/kaggle/working/*.png"): z.write(f, os.path.basename(f))
print(f"\n[COMPLETED] Successfully created archive: {zip_out} ({os.path.getsize(zip_out)/(1024*1024):.2f} MB)")
print("You can download 'ablations_5fold_results.zip' from Kaggle Output tab!")